In [0]:
# ============================================================
# Notebook: 06_optimization/01_liquid_clustering_benchmark
# Purpose : Measure and document the impact of Liquid Clustering
#           across the three largest fact tables:
#             1. fact_observations (16.2M rows) — cluster_by [patient_key, date_key]
#             2. fact_medications  (4.2M rows)  — cluster_by [patient_key, start_date_key]
#             3. fact_encounters   (3.2M rows)  — cluster_by [patient_key, start_date_key]
#           Each cell output is preserved for permanent documentation.
#
# Execution Order:
#   Cmd 1   : Define benchmark utility function (Run once at start).
#
#   Cmd 2-3 : fact_observations — Before & After (already completed).
#
#   Cmd 4   : fact_medications — "Before" (Run BEFORE applying cluster_by).
#   Cmd 5   : fact_encounters  — "Before" (Run BEFORE applying cluster_by).
#   ---     : Apply cluster_by in DLT notebooks & trigger Full Refresh.
#   Cmd 6   : fact_medications — "After" (Run AFTER Full Refresh).
#   Cmd 7   : fact_encounters  — "After" (Run AFTER Full Refresh).
#
#   Cmd 8   : Final Benchmark Summary — All three tables compared.
# ============================================================

In [0]:
# Cmd 1: Shared Benchmark Utility Function (Definition only)
import time
from pyspark.sql import functions as F

LOG_TABLE = "covid19_lakehouse.default.optimization_benchmark_log"


def run_benchmark(phase: str, table: str = "covid19_lakehouse.gold.fact_observations", date_col: str = "date_key"):
    """Run a patient + date range benchmark on the given fact table.

    Args:
        phase:    Label for the benchmark phase (e.g. "before_clustering_medications").
        table:    Fully qualified table name.
        date_col: Date column to filter on (date_key for observations,
                  start_date_key for medications/encounters).
    """
    # Select the patient with the highest record count for benchmarking
    sample_patient = (
        spark.table(table)
        .groupBy("patient_key")
        .count()
        .orderBy(F.desc("count"))
        .limit(1)
        .collect()[0]["patient_key"]
    )
    print(f" Sample patient_key: {sample_patient} | Phase: {phase} | Table: {table.split('.')[-1]}")

    # Build filtered query dataframe — patient + 2019 date range
    query_df = spark.table(table).filter(
        (F.col("patient_key") == sample_patient)
        & (F.col(date_col).between(20190101, 20191231))
    )

    # Execute query 3 times to measure average time
    times = []
    for i in range(3):
        start = time.time()
        row_count = query_df.count()
        times.append(time.time() - start)
        print(f"    Run {i + 1}: {times[-1]:.3f}s | Rows: {row_count:,}")

    avg_time = sum(times) / len(times)
    num_files_read = len(query_df.inputFiles())
    total_files = (
        spark.sql(f"DESCRIBE DETAIL {table}").select("numFiles").collect()[0]["numFiles"]
    )

    print(f"\n Avg time: {avg_time:.3f}s | Files read: {num_files_read}/{total_files}")

    # Log metrics into Delta table for persistent tracking
    spark.sql(
        f"CREATE TABLE IF NOT EXISTS {LOG_TABLE} "
        "(phase STRING, avg_time_sec DOUBLE, files_read INT, total_files INT, logged_at TIMESTAMP) "
        "USING DELTA"
    )
    spark.sql(f"DELETE FROM {LOG_TABLE} WHERE phase = '{phase}'")
    spark.createDataFrame(
        [(phase, avg_time, num_files_read, total_files)],
        schema="phase STRING, avg_time_sec DOUBLE, files_read INT, total_files INT",
    ).withColumn("logged_at", F.current_timestamp()).write.format("delta").mode(
        "append"
    ).saveAsTable(LOG_TABLE)

    print(f" Metric logged successfully under phase: '{phase}'")


In [0]:
# Cmd 2: 🔵 "Before" Phase - Execute this cell BEFORE modifying fact_observations
run_benchmark("before_clustering_observations")

 Sample patient_key: -4363131564242424850 | Phase: before_clustering
    Run 1: 2.189s | Rows: 422
    Run 2: 1.332s | Rows: 422
    Run 3: 1.342s | Rows: 422

 Avg time: 1.621s | Files read: 1/1
 Metric logged successfully under phase: 'before_clustering'


In [0]:
# Cmd 3: 🟢 "After" Phase - Execute this cell AFTER applying cluster_by + Full Refresh
run_benchmark("after_clustering_observations")

 Sample patient_key: -4363131564242424850 | Phase: after_clustering
    Run 1: 1.019s | Rows: 422
    Run 2: 0.785s | Rows: 422
    Run 3: 0.766s | Rows: 422

 Avg time: 0.857s | Files read: 1/1
 Metric logged successfully under phase: 'after_clustering'


In [0]:
# Cmd 4: 🔵 fact_medications — "Before" Phase
# Execute this cell BEFORE applying cluster_by to fact_medications.
run_benchmark("before_clustering_medications", table="covid19_lakehouse.gold.fact_medications", date_col="start_date_key")

 Sample patient_key: 7455112458596277877 | Phase: before_clustering_medications | Table: fact_medications
    Run 1: 1.670s | Rows: 51
    Run 2: 0.746s | Rows: 51
    Run 3: 0.861s | Rows: 51

 Avg time: 1.092s | Files read: 1/1
 Metric logged successfully under phase: 'before_clustering_medications'


In [0]:
# Cmd 5: 🔵 fact_encounters — "Before" Phase
# Execute this cell BEFORE applying cluster_by to fact_encounters.
run_benchmark("before_clustering_encounters", table="covid19_lakehouse.gold.fact_encounters", date_col="start_date_key")

 Sample patient_key: 6524076807236078949 | Phase: before_clustering_encounters | Table: fact_encounters
    Run 1: 0.983s | Rows: 93
    Run 2: 0.619s | Rows: 93
    Run 3: 0.694s | Rows: 93

 Avg time: 0.765s | Files read: 1/1
 Metric logged successfully under phase: 'before_clustering_encounters'


In [0]:
# Cmd 6: 🟢 fact_medications — "After" Phase
# Execute this cell AFTER applying cluster_by to fact_medications + Full Refresh.
run_benchmark("after_clustering_medications", table="covid19_lakehouse.gold.fact_medications", date_col="start_date_key")

 Sample patient_key: 7455112458596277877 | Phase: after_clustering_medications | Table: fact_medications
    Run 1: 0.816s | Rows: 51
    Run 2: 0.799s | Rows: 51
    Run 3: 0.731s | Rows: 51

 Avg time: 0.782s | Files read: 1/1
 Metric logged successfully under phase: 'after_clustering_medications'


In [0]:
# Cmd 7: 🟢 fact_encounters — "After" Phase
# Execute this cell AFTER applying cluster_by to fact_encounters + Full Refresh.
run_benchmark("after_clustering_encounters", table="covid19_lakehouse.gold.fact_encounters", date_col="start_date_key")

 Sample patient_key: 6524076807236078949 | Phase: after_clustering_encounters | Table: fact_encounters
    Run 1: 0.612s | Rows: 93
    Run 2: 0.635s | Rows: 93
    Run 3: 0.648s | Rows: 93

 Avg time: 0.632s | Files read: 1/1
 Metric logged successfully under phase: 'after_clustering_encounters'


In [0]:
# Cmd 8:  Final Comparison Summary — All Three Fact Tables
# Execute after completing all Before/After phases for all tables.
import pandas as pd

results = spark.table(LOG_TABLE).orderBy("logged_at").toPandas()

print("=" * 72)
print(" LIQUID CLUSTERING BENCHMARK — BEFORE vs AFTER COMPARISON")
print("=" * 72)
print(results[["phase", "avg_time_sec", "files_read", "total_files"]].to_string(index=False))

# Define table groups for comparison
bench_groups = {
    "fact_observations": ("before_clustering_observations", "after_clustering_observations"),
    "fact_medications": ("before_clustering_medications", "after_clustering_medications"),
    "fact_encounters": ("before_clustering_encounters", "after_clustering_encounters"),
}

print("=" * 72)
print(f"{'Table':<25} {'Before (s)':>12} {'After (s)':>12} {'Improvement':>14}")
print("-" * 72)

for tbl, (before_phase, after_phase) in bench_groups.items():
    if {before_phase, after_phase}.issubset(set(results["phase"])):
        before = results[results["phase"] == before_phase].iloc[-1]
        after = results[results["phase"] == after_phase].iloc[-1]
        improvement = (1 - after["avg_time_sec"] / before["avg_time_sec"]) * 100
        print(f"{tbl:<25} {before['avg_time_sec']:>12.3f} {after['avg_time_sec']:>12.3f} {improvement:>13.1f}%")
    else:
        print(f"{tbl:<25} {'pending':>12} {'pending':>12} {'pending':>14}")

print("=" * 72)

 LIQUID CLUSTERING BENCHMARK — BEFORE vs AFTER COMPARISON
                         phase  avg_time_sec  files_read  total_files
before_clustering_observations      1.620842           1            1
 after_clustering_observations      0.856509           1            1
 before_clustering_medications      1.092154           1            1
  before_clustering_encounters      0.765255           1            1
  after_clustering_medications      0.781944           1            1
   after_clustering_encounters      0.631707           1            1
Table                       Before (s)    After (s)    Improvement
------------------------------------------------------------------------
fact_observations                1.621        0.857          47.2%
fact_medications                 1.092        0.782          28.4%
fact_encounters                  0.765        0.632          17.5%
